# 04 - 存储层次（Memory Hierarchy）

这个 notebook 用 Python 模拟**缓存、内存、硬盘**的分级存储系统。你将亲手操作 Cache，亲眼看到局部性原理带来的性能差异。

## 怎么用？
从上到下依次运行每个 cell（`Shift + Enter`），观察输出，动手改参数实验。

## 今日学习路线

| # | 内容 |
|---|------|
| 4.1 | 存储金字塔可视化 |
| 4.2 | Cache 模拟器：直接映射 |
| 4.3 | 组相联 Cache |
| 4.4 | 局部性实验 |
| 4.5 | 写策略对比 |
| 4.6 | 虚拟内存演示 |

---
## 4.1 存储金字塔可视化

先来直观感受一下各级存储的速度和容量差异。

In [ ]:
# ============================================
# 存储金字塔
# ============================================

import time

def memory_pyramid():
    """展示存储层级的速度和容量差异"""
    levels = [
        ("寄存器",     0.3,    "~1 KB",      "CPU 内部", "⚡"),
        ("L1 Cache",   1,      "~64 KB",     "CPU 内部", "🔵"),
        ("L2 Cache",   7,      "~512 KB",    "CPU 内部", "🟢"),
        ("L3 Cache",   30,     "~16 MB",     "片外/共享", "🟡"),
        ("主存 RAM",   100,    "~16 GB",     "DIMM 插槽", "🟠"),
        ("SSD",        100_000, "~1 TB",     "M.2/SATA", "🔴"),
        ("HDD",        10_000_000, "~10 TB", "磁碟", "🐢"),
    ]
    
    print("存储金字塔（从顶向下，越来越慢）\n")
    print(f"{'层级':<12s} | {'访问延迟':>12s} | {'容量':>12s} | {'位置':<12s}")
    print("-" * 60)
    
    baseline = levels[0][1]
    for name, latency, capacity, location, emoji in levels:
        # 格式化延迟
        if latency < 1000:
            lat_str = f"{latency:.1f} ns"
        elif latency < 1_000_000:
            lat_str = f"{latency/1000:.0f} μs"
        else:
            lat_str = f"{latency/1_000_000:.0f} ms"
        
        ratio = latency / baseline
        bar = '█' * min(int(ratio), 40)
        print(f"{emoji} {name:<10s} | {lat_str:>12s} | {capacity:>12s} | {location:<12s} {bar}")
    
    print(f"\n💡 从寄存器到 HDD，速度差: {levels[-1][1] / levels[0][1]:,.0f} 倍")
    print(f"💡 如果寄存器访问是 1 秒，HDD 就是 {levels[-1][1] / levels[0][1] * 0.3 / 1e9:.0f} 年")

memory_pyramid()

---
## 4.2 Cache 模拟器

下面实现一个完整的 Cache 模拟器，支持不同映射方式，追踪命中率。

In [ ]:
# ============================================
# Cache 模拟器
# ============================================

class CacheSimulator:
    """
    配置参数:
        cache_size: Cache 总大小（字节）
        block_size: 每个 Cache Line 大小（字节）
        associativity: 相联度（1=直接映射, 全大小=全相联, N=组相联）
    """
    
    def __init__(self, cache_size=256, block_size=16, associativity=1, name="Cache"):
        self.name = name
        self.cache_size = cache_size
        self.block_size = block_size
        self.associativity = associativity
        
        self.num_blocks = cache_size // block_size
        self.num_sets = self.num_blocks // associativity
        
        # 每个槽位: (valid, tag, data, last_access_time, dirty)
        self.slots = []
        for _ in range(self.num_blocks):
            self.slots.append({
                'valid': False,
                'tag': 0,
                'data': [0] * block_size,
                'last_access': 0,
                'dirty': False
            })
        
        # 统计
        self.hits = 0
        self.misses = 0
        self.access_time = 0  # 累计访问时间
    
    def _get_set_index_and_tag(self, address):
        """从地址解析出 (组索引, Tag, 块内偏移)"""
        block_number = address // self.block_size
        set_index = block_number % self.num_sets
        tag = block_number // self.num_sets
        offset = address % self.block_size
        return set_index, tag, offset
    
    def read(self, address):
        """读取一个地址，返回 (值, 是否命中)"""
        set_idx, tag, offset = self._get_set_index_and_tag(address)
        
        # 在组内查找
        start_slot = set_idx * self.associativity
        end_slot = start_slot + self.associativity
        
        self.access_time += 1
        
        for i in range(start_slot, end_slot):
            slot = self.slots[i]
            if slot['valid'] and slot['tag'] == tag:
                # Cache HIT!
                self.hits += 1
                slot['last_access'] = self.access_time
                return slot['data'][offset], True
        
        # Cache MISS — 从内存加载整个块
        self.misses += 1
        self.access_time += 100  # 内存延迟惩罚
        
        # 找到一个槽位（优先无效的，否则 LRU 替换）
        target_slot_idx = None
        for i in range(start_slot, end_slot):
            if not self.slots[i]['valid']:
                target_slot_idx = i
                break
        
        if target_slot_idx is None:
            # 需要替换 —— LRU
            oldest_time = float('inf')
            for i in range(start_slot, end_slot):
                if self.slots[i]['last_access'] < oldest_time:
                    oldest_time = self.slots[i]['last_access']
                    target_slot_idx = i
            # 如果脏块，写回内存
            if self.slots[target_slot_idx]['dirty']:
                self.access_time += 100  # 写回惩罚
        
        # 加载新块
        slot = self.slots[target_slot_idx]
        slot['valid'] = True
        slot['tag'] = tag
        slot['last_access'] = self.access_time
        slot['dirty'] = False
        # 模拟"从内存读数据"
        block_start = (address // self.block_size) * self.block_size
        for j in range(self.block_size):
            slot['data'][j] = (block_start + j) % 256  # 伪数据
        
        return slot['data'][offset], False
    
    def write(self, address, value, write_back=True):
        """写入一个地址"""
        set_idx, tag, offset = self._get_set_index_and_tag(address)
        start_slot = set_idx * self.associativity
        end_slot = start_slot + self.associativity
        
        self.access_time += 1
        
        # 查找
        for i in range(start_slot, end_slot):
            slot = self.slots[i]
            if slot['valid'] and slot['tag'] == tag:
                self.hits += 1
                slot['data'][offset] = value
                slot['last_access'] = self.access_time
                if write_back:
                    slot['dirty'] = True
                else:
                    self.access_time += 100  # Write-Through 惩罚
                return True
        
        # Write Miss
        self.misses += 1
        # 简化：先加载再写 (Write-Allocate)
        self.read(address)
        self.write(address, value, write_back)
        return False
    
    def stats(self):
        """打印统计"""
        total = self.hits + self.misses
        hit_rate = self.hits / total * 100 if total > 0 else 0
        print(f"\n📊 {self.name} 统计:")
        print(f"  配置: {self.cache_size}B, {self.block_size}B/块, {self.associativity}路组相联")
        print(f"  访问次数: {total}")
        print(f"  命中(Hit):  {self.hits}  ({hit_rate:.1f}%)")
        print(f"  缺失(Miss): {self.misses}  ({100-hit_rate:.1f}%)")
        print(f"  累计周期: {self.access_time}")
        avg_lat = self.access_time / total if total > 0 else 0
        print(f"  平均延迟: {avg_lat:.1f} 周期/访问")
        return hit_rate

# 测试
cache = CacheSimulator(cache_size=128, block_size=16, associativity=1, name="直接映射Cache")
# 顺序访问
for addr in range(0, 64):
    cache.read(addr)
cache.stats()

---
## 4.3 直接映射 vs 组相联

同一个访问序列，不同 Cache 配置的表现截然不同。

In [ ]:
# ============================================
# 对比不同映射方式的命中率
# ============================================

def compare_cache_configs(access_pattern, pattern_name):
    """对比不同 Cache 配置在同一访问模式下的表现"""
    configs = [
        ("直接映射 (1路)", 128, 16, 1),
        ("2路组相联",     128, 16, 2),
        ("4路组相联",     128, 16, 4),
        ("全相联 (8路)",   128, 16, 8),
    ]
    
    print(f"📊 访问模式: {pattern_name}\n")
    print(f"{'配置':<25s} | {'命中率':>8s} | {'平均延迟':>10s}")
    print("-" * 50)
    
    for name, size, block, assoc in configs:
        c = CacheSimulator(size, block, assoc, name)
        for addr in access_pattern:
            c.read(addr)
        
        total = c.hits + c.misses
        hit_rate = c.hits / total * 100 if total > 0 else 0
        avg_lat = c.access_time / total if total > 0 else 0
        print(f"{name:<25s} | {hit_rate:7.1f}% | {avg_lat:8.1f} 周期")

# 测试1: 顺序访问（空间局部性好）
sequential = list(range(0, 128))
compare_cache_configs(sequential, "顺序访问 0→127")

# 测试2: 跨步访问 — 故意制造冲突
print("\n")
# 每隔 64 字节一跳（如果块=16B，64字节 = 4块间隔，可能导致直接映射冲突）
strided = []
for i in range(32):
    strided.append(i * 16)       # 0, 16, 32, ...
    strided.append(i * 16 + 128)  # 128, 144, ...  (可能冲突！)
compare_cache_configs(strided, "跨步访问 (16B步长，可能冲突)")

# 测试3: 完全随机
print("\n")
import random
random.seed(42)
random_access = [random.randint(0, 255) for _ in range(100)]
compare_cache_configs(random_access, "随机访问 (100 次)")

### 🔍 观察

- **顺序访问**时，所有配置表现都好——空间局部性让预取很有效
- **冲突访问**时，直接映射退化严重（两个地址映射到同一个槽位 → ping-pong 效应），组相联和全相联表现好得多
- **随机访问**时，所有配置都不理想——算法层面的改进最重要

---
## 4.4 局部性实验：矩阵遍历

同一个矩阵，行优先 vs 列优先遍历，性能差距有多大？

In [ ]:
# ============================================
# 矩阵遍历——行优先 vs 列优先
# ============================================

import numpy as np
import time as time_module

# 用我们的 Cache 模拟地址访问序列

def simulate_matrix_traversal(rows, cols, row_major=True):
    """
    模拟矩阵遍历的 cache 行为
    假设矩阵以行优先存储在内存中（如 C/C++）
    base_addr = 0, 每个元素 4 字节
    """
    elem_size = 4
    cache = CacheSimulator(cache_size=512, block_size=64, associativity=4, name="Matrix")
    
    if row_major:
        for r in range(rows):
            for c in range(cols):
                addr = (r * cols + c) * elem_size
                cache.read(addr)
    else:
        for c in range(cols):
            for r in range(rows):
                addr = (r * cols + c) * elem_size
                cache.read(addr)
    
    return cache

# 100x100 矩阵
ROWS, COLS = 100, 100

cache_row = simulate_matrix_traversal(ROWS, COLS, row_major=True)
cache_col = simulate_matrix_traversal(ROWS, COLS, row_major=False)

print("📊 矩阵 100×100, 元素 4 字节, Cache 512B, 64B/块, 4路\n")

print(f"{'访问模式':<20s} | {'命中次数':>10s} | {'缺失次数':>10s} | {'命中率':>8s} | {'总周期':>10s}")
print("-" * 70)
for name, c in [("行优先 (Cache友好)", cache_row), ("列优先 (Cache不友好)", cache_col)]:
    total = c.hits + c.misses
    hit_rate = c.hits / total * 100
    print(f"{name:<20s} | {c.hits:10d} | {c.misses:10d} | {hit_rate:7.1f}% | {c.access_time:10d}")

speedup = cache_col.access_time / cache_row.access_time
print(f"\n⚡ 行优先快 {speedup:.1f} 倍！")
print(f"\n💡 原因: 行优先遍历时，同一 Cache Line(64B) 中的连续元素被连续访问")
print(f"   列优先遍历时，每次访问跳到下一行（跳过 100*4=400 字节），Cache Line 浪费")

In [ ]:
# ============================================
# 用真实 numpy 验证
# ============================================

def real_matrix_sum_test(size=2000):
    """用 numpy 实测行优先 vs 列优先的差异"""
    A = np.random.randn(size, size).astype(np.float64)
    
    # 行优先求和
    start = time_module.perf_counter()
    row_sum = 0.0
    for i in range(size):
        for j in range(size):
            row_sum += A[i, j]
    row_time = time_module.perf_counter() - start
    
    # 列优先求和
    start = time_module.perf_counter()
    col_sum = 0.0
    for j in range(size):
        for i in range(size):
            col_sum += A[i, j]
    col_time = time_module.perf_counter() - start
    
    print(f"📊 真实 numpy 测试: {size}×{size} 矩阵求和\n")
    print(f"  行优先 (i→j): {row_time:.3f}s")
    print(f"  列优先 (j→i): {col_time:.3f}s")
    print(f"  列优先慢了: {col_time / row_time:.2f}x")
    print(f"  ✅ 结果一致: {abs(row_sum - col_sum) < 0.01}")

# 用小一点避免 notebook 跑太久
real_matrix_sum_test(1500)

### ✏️ 自己动手

修改上面的 size 参数（比如 500, 1000, 2000），观察不同大小的加速比变化。为什么矩阵越大，行优先的优势越明显？

---
## 4.5 写策略对比

Write-Through vs Write-Back，模拟它们的差异。

In [ ]:
# ============================================
# Write-Through vs Write-Back
# ============================================

def simulate_write_policies():
    """模拟频繁写操作的场景"""
    
    # 场景：对同一个地址频繁写入（时间局部性极强）
    # 例如 sum 变量在循环中被反复更新
    
    addr = 0x100
    iterations = 100
    
    print("📊 场景: 对同一地址连续写入 100 次\n")
    print(f"{'策略':<20s} | {'Hit':>7s} | {'Miss':>7s} | {'总周期':>10s} | {'说明'}")
    print("-" * 65)
    
    # Write-Back
    cache_wb = CacheSimulator(128, 16, 1, "WB")
    for i in range(iterations):
        cache_wb.write(addr, i, write_back=True)
    total_wb = cache_wb.hits + cache_wb.misses
    print(f"{'Write-Back':<20s} | {cache_wb.hits:7d} | {cache_wb.misses:7d} | {cache_wb.access_time:10d} | 只写Cache, 标Dirty")
    print(f"{'':20s} | {'命中率':>7s}: {cache_wb.hits/total_wb*100:.0f}%")
    print()
    
    # Write-Through
    cache_wt = CacheSimulator(128, 16, 1, "WT")
    for i in range(iterations):
        cache_wt.write(addr, i, write_back=False)  # Write-Through
    total_wt = cache_wt.hits + cache_wt.misses
    print(f"{'Write-Through':<20s} | {cache_wt.hits:7d} | {cache_wt.misses:7d} | {cache_wt.access_time:10d} | 每次写同时更新内存")
    print(f"{'':20s} | {'命中率':>7s}: {cache_wt.hits/total_wt*100:.0f}%")
    
    speedup = cache_wt.access_time / cache_wb.access_time
    print(f"\n⚡ Write-Back 快 {speedup:.1f}x")
    print(f"💡 原因: Write-Back 的重复写都在 Cache 内完成，不触发内存访问")
    print(f"💡 代价: 如果突然断电，Write-Back 可能丢数据（Dirty 块未写回）")

simulate_write_policies()

---
## 4.6 虚拟内存模拟

模拟页表翻译和 TLB 加速效果。

In [ ]:
# ============================================
# 虚拟内存：页表 + TLB
# ============================================

class MMU:
    """内存管理单元：支持分页 + TLB"""
    
    def __init__(self, page_size=4096, tlb_entries=16, physical_pages=256):
        self.page_size = page_size
        self.physical_pages = physical_pages
        
        # 页表: VPN → (valid, PFN)
        self.page_table = {}
        
        # TLB: 小型高速缓存 [(VPN, PFN, valid)]
        self.tlb = []
        self.tlb_max = tlb_entries
        
        # 统计
        self.tlb_hits = 0
        self.tlb_misses = 0
        self.page_faults = 0
        
        # 预建一些映射
        for vpn in range(50):
            self.page_table[vpn] = (True, vpn + 10)  # 映射到 VPN+10
    
    def translate(self, virtual_addr):
        """虚拟地址 → 物理地址"""
        vpn = virtual_addr // self.page_size
        offset = virtual_addr % self.page_size
        
        # 1. 先查 TLB
        for entry in self.tlb:
            if entry['vpn'] == vpn and entry['valid']:
                self.tlb_hits += 1
                pfn = entry['pfn']
                return pfn * self.page_size + offset, "TLB Hit"
        
        # 2. TLB Miss → 查页表
        self.tlb_misses += 1
        
        if vpn in self.page_table:
            valid, pfn = self.page_table[vpn]
            if valid:
                # 加载到 TLB
                self._tlb_insert(vpn, pfn)
                return pfn * self.page_size + offset, "TLB Miss (页表命中)"
        
        # 3. Page Fault — 不在物理内存中
        self.page_faults += 1
        # 分配新物理页
        new_pfn = len(self.page_table) + 10
        self.page_table[vpn] = (True, new_pfn)
        self._tlb_insert(vpn, new_pfn)
        return new_pfn * self.page_size + offset, "Page Fault (从磁盘加载)"
    
    def _tlb_insert(self, vpn, pfn):
        """插入 TLB，如果满了就踢掉最旧的"""
        if len(self.tlb) >= self.tlb_max:
            self.tlb.pop(0)  # FIFO 替换
        self.tlb.append({'vpn': vpn, 'pfn': pfn, 'valid': True})
    
    def stats(self):
        total = self.tlb_hits + self.tlb_misses
        if total > 0:
            print(f"\n📊 MMU 统计:")
            print(f"  页大小: {self.page_size} B")
            print(f"  TLB 条目: {self.tlb_max}")
            print(f"  TLB 命中: {self.tlb_hits} ({self.tlb_hits/total*100:.1f}%)")
            print(f"  TLB 缺失: {self.tlb_misses} ({self.tlb_misses/total*100:.1f}%)")
            print(f"  Page Fault: {self.page_faults}")

# 测试
mmu = MMU()

print("🔍 地址翻译演示:\n")
print(f"{'虚拟地址':>12s} → {'物理地址':>12s} | {'来源':<25s}")
print("-" * 55)

random.seed(42)
accesses = [100, 5000, 100, 20000, 5000, 100, 88888, 100, 5000]
for va in accesses:
    pa, source = mmu.translate(va)
    print(f"  0x{va:08X}  →  0x{pa:08X}  | {source}")

mmu.stats()
print(f"\n💡 注意: 地址 100 第二次访问时 TLB Hit（时间局部性起效！）")

---
## 🎯 综合挑战

In [ ]:
# Q1: 给定以下访问序列，预测直接映射 Cache (128B, 16B/块) 的命中率
# 访问序列: 0, 16, 32, 48, 0, 16, 32, 48

cache_q1 = CacheSimulator(128, 16, 1)
for addr in [0, 16, 32, 48, 0, 16, 32, 48]:
    cache_q1.read(addr)

total = cache_q1.hits + cache_q1.misses
print(f"Q1: 命中 {cache_q1.hits}/{total} = {cache_q1.hits/total*100:.0f}%")
print(f"   为什么后半段全部命中？→ 时间局部性 + Cache 够大")

In [ ]:
# Q2: 如果一个 Cache Line = 64 字节，访问地址 1000 时，
#     哪些地址的数据会被一起加载进 Cache？

addr = 1000
block_size = 64
block_start = (addr // block_size) * block_size
block_end = block_start + block_size - 1

print(f"Q2: 访问地址 {addr}")
print(f"   所在 Cache Line: [{block_start}, {block_end}]")
print(f"   → 这 64 个字节都被加载（空间局部性）")
print(f"   → 接下来如果访问 {block_start+4}，Cache Hit！")

In [ ]:
# Q3: 以下哪种访问模式对 Cache 最友好？
# A) 随机跳来跳去
# B) 顺序访问数组
# C) 每隔一大段访一次
# D) 倒序访问

patterns = {
    'A 随机': [random.randint(0, 255) for _ in range(64)],
    'B 顺序': list(range(0, 128, 2)),      # 64次
    'C 大步跳跃': [i * 32 for i in range(64)],  # 步长32 = 可能每次都miss
    'D 倒序': list(range(126, -1, -2)),
}

print("Q3: 不同访问模式的 Cache 命中率\n")
for name, pattern in patterns.items():
    c = CacheSimulator(128, 16, 1)
    for addr in pattern:
        c.read(addr)
    total = c.hits + c.misses
    print(f"  {name}: {c.hits/total*100:.0f}%")

print("\n✅ 结论: 顺序访问 > 倒序访问 > 大步跳跃 > 随机")

---
## ✅ 检查清单

对照一下，都掌握了吗？

- [ ] 存储金字塔：各层级的速度和容量量级
- [ ] 局部性原理：时间局部性 vs 空间局部性
- [ ] Cache Line / Block 的概念
- [ ] 三种映射方式：直接映射、全相联、组相联
- [ ] 替换策略：LRU、FIFO、Random
- [ ] 写策略：Write-Through vs Write-Back
- [ ] 虚拟内存：分页、页表、MMU、TLB
- [ ] 如何在编程时利用局部性提升性能

---

> 📖 下一章：[05 - 流水线与并行](../05-pipeline-and-parallelism/) — 如何让 CPU 同时执行多条指令